# Test data.py and tools.py

In [ ]:
# import sys
# sys.path.insert(0, "../src")

In [2]:
# from cx_agent.data import load_fabsa, VALID_CHILD_ASPECTS, VALID_INDUSTRIES
# from cx_agent.tools import describe, infer, report

In [ ]:
# # test data loader
# df_reviews, df_exploded = load_fabsa()
# print(f"Reviews: {len(df_reviews)}")   # expect 10574
# print(f"Labels:  {len(df_exploded)}")  # expect 18668

Reviews: 10574
Labels:  18668


In [ ]:
# # test describe tool
# result = describe(df_exploded, industry="Banking", sentiment="negative", top_n=5)
# print(result)

{'total_labels': 632, 'unique_reviews': 318, 'sentiment_distribution': {'negative': 1.0}, 'top': [{'child_aspect': 'app-website', 'sentiment': 'negative', 'count': 146}, {'child_aspect': 'attitude-of-staff', 'sentiment': 'negative', 'count': 109}, {'child_aspect': 'ease-of-use', 'sentiment': 'negative', 'count': 107}, {'child_aspect': 'account-access', 'sentiment': 'negative', 'count': 102}, {'child_aspect': 'general-satisfaction', 'sentiment': 'negative', 'count': 76}], 'filters_applied': {'industry': 'Banking', 'data_source': None, 'parent_aspect': None, 'child_aspect': None, 'sentiment': 'negative'}}


In [ ]:
# # test infer tool
# result = infer(df_exploded, compare_field="child_aspect", group_a="app-website", group_b="speed")
# print(result)

{'group_a': {'name': 'app-website', 'n': 4874, 'sentiment': {'positive': 2670, 'negative': 1842, 'neutral': 362}}, 'group_b': {'name': 'speed', 'n': 1310, 'sentiment': {'positive': 941, 'negative': 367, 'neutral': 2}}, 'test': 'chi-squared', 'chi2': np.float64(171.866), 'p_value': np.float64(0.0), 'significant': True, 'warnings': []}


In [ ]:
# # test report tool
# result = report(df_exploded, df_reviews, industry="Fashion")
# print(result)

{'scope': {'industry': 'Fashion', 'data_source': None}, 'total_reviews': 2846, 'total_labels': 4812, 'avg_labels_per_review': 1.7, 'avg_words_per_review': np.float64(17.0), 'sentiment_distribution': {'positive': 0.728, 'negative': 0.27, 'neutral': 0.002}, 'top_negative_aspects': {'app-website': 381, 'general-satisfaction': 221, 'attitude-of-staff': 168}, 'top_positive_aspects': {'general-satisfaction': 1147, 'app-website': 779, 'ease-of-use': 560}, 'sample_verbatims': {'positive': {'text': 'App is very simple and easy to use!', 'aspect': 'purchase-booking-experience.ease-of-use'}, 'negative': {'text': 'Chat to claim an order not delivered for a month since no customer service but the application closes the conversation 30 seconds after it is shameful everything is done for us to give up!', 'aspect': 'logistics-rides.speed'}}}


In [ ]:
# # test invalid industry
# try:
#     describe(df_exploded, industry="InvalidIndustry")
# except ValueError as e:
#     print(f"Caught expected error: {e}")

Caught expected error: Invalid industry: 'InvalidIndustry'. Must be one of: ['Banking', 'Consulting', 'Fashion', 'Groceries', 'Information Technology', 'Price Comparison', 'Ride Hailing', 'Streaming', 'Trading', 'Travel Booking']


# Test llm.py

In [2]:
from cx_agent.llm import chat

In [ ]:
# # test tool selection
# response = chat([{"role": "user", "content": "What are the top complaints in Banking?"}])

# msg = response.choices[0].message
# print(f"Content: {msg.content}")
# print(f"Tool calls: {msg.tool_calls}")

# if msg.tool_calls:
#     tc = msg.tool_calls[0]
#     print(f"\nTool chosen: {tc.function.name}")
#     print(f"Arguments:  {tc.function.arguments}")

Content: None
Tool calls: [ChatCompletionMessageToolCall(function=Function(arguments='{"group_by": "child_aspect", "industry": "Banking", "sentiment": "negative", "top_n": 5}', name='describe'), id='call_7479fc8d-370d-42b2-9a29-100feec03a65', type='function')]

Tool chosen: describe
Arguments:  {"group_by": "child_aspect", "industry": "Banking", "sentiment": "negative", "top_n": 5}


In [3]:
# test all 3 question types
test_questions = [
    "What are the top complaints in Price Comparison?",
    "Is sentiment significantly different between app-website and speed?",
    "Give me a summary report for Fashion",
]

for q in test_questions:
    response = chat([{"role": "user", "content": q}])
    msg = response.choices[0].message
    if msg.tool_calls:
        tc = msg.tool_calls[0]
        print(f"Q: {q}")
        print(f"→ Tool: {tc.function.name} | Args: {tc.function.arguments}\n")
    else:
        print(f"Q: {q}")
        print(f"→ No tool called. Content: {msg.content}\n")

Q: What are the top complaints in Price Comparison?
→ Tool: describe | Args: {"industry": "Price Comparison", "sentiment": "negative"}

Q: Is sentiment significantly different between app-website and speed?
→ Tool: infer | Args: {"compare_field": "child_aspect", "group_a": "app-website", "group_b": "speed"}

Q: Give me a summary report for Fashion
→ Tool: report | Args: {"industry": "Fashion"}



# Test agent.py

In [5]:
import sys
sys.path.insert(0, "../src")

In [6]:
from cx_agent.agent import run
import json

In [3]:
# # run full agent loop
# result = run(
#     "What are the top complaints in Banking?")

In [7]:
# # test all 3 question types
# questions = [
#     "What are the top complaints in Banking?",
#     "Is there a significant difference in sentiment between app-website and speed?",
#     "Give me a summary report for the Fashion industry",
# ]

# for q in questions:
#     print(f"\n{'='*60}")
#     print(f"Q: {q}")
#     result = run(q, verbose=False)
#     print(f"A: {result['answer']}")
#     print(f"Steps: {len(result['trace'])}")


Q: What are the top complaints in Banking?
A: The top complaints in Banking are app-website (146 mentions), attitude-of-staff (109), ease-of-use (107), account-access (102), and general-satisfaction (76).
Steps: 2

Q: Is there a significant difference in sentiment between app-website and speed?
A: There is a significant difference in sentiment between app-website and speed aspects, with more negative sentiments for app-website (1842 mentions) compared to speed (367 mentions).
Steps: 2

Q: Give me a summary report for the Fashion industry
A: The Fashion industry has 2,846 reviews with a positive sentiment of 72.8%. The most praised aspects are app-website (779 mentions) and general-satisfaction (1,147 mentions). Common complaints include app-website issues (381 mentions), general satisfaction (221 mentions), and staff attitude (168 mentions).
Steps: 2


In [3]:
# test out of scope
result = run("What is the capital of France?")


--- Step 1 ---
Tool: describe
Args: {'industry': '', 'data_source': '', 'parent_aspect': '', 'child_aspect': '', 'sentiment': '', 'top_n': 0}
Result: {
  "total_labels": 18668,
  "unique_reviews": 10574,
  "sentiment_distribution": {
    "positive": 0.652,
    "negative": 0.312,
    "neutral": 0.036
  },
  "top": [],
  "filters_applied": {
    "industry": "",
    "data_source": "",
    "parent_aspect": "",
    "child_aspect": "",
    "sentiment": ""
  }
}

--- Step 2 ---
Answer: I can only answer questions about customer feedback data from the FABSA dataset. Please ask about industries, aspects, or sentiment.


In [7]:
# test another
result = run("what is the top trending shoes with best reviews?")


--- Step 1 ---
Tool: describe
Args: {'group_by': 'child_aspect', 'industry': 'Fashion', 'data_source': 'Trustpilot', 'sentiment': 'positive', 'top_n': 1}
Result: {
  "total_labels": 632,
  "unique_reviews": 373,
  "sentiment_distribution": {
    "positive": 1.0
  },
  "top": [
    {
      "child_aspect": "general-satisfaction",
      "sentiment": "positive",
      "count": 277
    }
  ],
  "filters_applied": {
    "industry": "Fashion",
    "data_source": "Trustpilot",
    "parent_aspect": null,
    "child_aspect": null,
    "sentiment": "positive"
  }
}

--- Step 2 ---
Answer: The top trending shoes with the best reviews are related to general satisfaction, mentioned positively in 277 reviews.
